# Microsoft Agent Framework

## AI-powered agent that automatically converts expense data into a formatted claim and submits it to the finance team using a tool

Organizations often require employees to submit expense claims for reimbursements such as travel, meals, or accommodation. Typically, this process involves manually reviewing expense data, formatting it into a structured claim, calculating totals, and sending it to the finance department via email.

Manual handling of this process can lead to:

- Time-consuming claim preparation
- Human errors in calculations
- Inconsistent formatting of expense reports
- Delays in claim submission

To address this problem, an AI-powered Expense Claim Submission Agent is implemented using an agent framework and Azure OpenAI.

The agent:

1. Reads expense data from a file.
2. Accepts a user request regarding the expense data.
3. Uses an AI model to interpret the request.
4. Automatically formats the expenses into an itemized claim.
5. Calculates the total expense.
6. Sends the formatted claim via an email tool to the finance department.

Confirms to the user that the claim has been submitted.
<img src="./Assests/flow.png" alt="images" width="350" style="display:block; margin-top:10px;"/>

In [1]:
%pip install python-dotenv azure-identity agent-framework==1.0.0rc2 opentelemetry-semantic-conventions-ai==0.4.13

Defaulting to user installation because normal site-packages is not writeable
  Using cached azure_ai_projects-2.0.0b3-py3-none-any.whl.metadata (68 kB)
Using cached azure_ai_projects-2.0.0b3-py3-none-any.whl (240 kB)
  Attempting uninstall: azure-ai-projects
    Found existing installation: azure-ai-projects 2.0.0b2
    Uninstalling azure-ai-projects-2.0.0b2:
      Successfully uninstalled azure-ai-projects-2.0.0b2
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\vijay\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
import asyncio
from pathlib import Path

In [3]:
# Add references
from agent_framework import Agent, tool
from agent_framework.azure import AzureOpenAIResponsesClient
from azure.identity import AzureCliCredential
from pydantic import Field
from typing import Annotated

In [4]:
from dotenv import load_dotenv

load_dotenv()

deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")

print("Deployment:", deployment)
print("Endpoint:", endpoint)

Deployment: gpt-4o
Endpoint: https://tavant-mf.services.ai.azure.com/api/projects/proj-default


In [5]:
expenses_data = """
Taxi - $35
Lunch - $18
Hotel - $220
Flight - $450
"""
print(expenses_data)


Taxi - $35
Lunch - $18
Hotel - $220
Flight - $450



In [6]:
@tool(approval_mode="never_require")
def submit_claim(
    to: Annotated[str, Field(description="Who to send the email to")],
    subject: Annotated[str, Field(description="The subject of the email.")],
    body: Annotated[str, Field(description="The text body of the email.")]
):
    print("\n--- EMAIL SENT ---")
    print("To:", to)
    print("Subject:", subject)
    print("Body:\n", body)

In [7]:
async def process_expenses_data(prompt, expenses_data):

    credential = AzureCliCredential()

    async with Agent(
        client=AzureOpenAIResponsesClient(
            credential=credential,
            deployment_name=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
            project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
        ),
        instructions="""
        You are an AI assistant for expense claim submission.

        At the user's request, create an expense claim and use the plug-in
        function to send an email to expenses@contoso.com with the subject
        'Expense Claim'.

        The email body should contain:
        - Itemized expenses
        - Total expense

        Then confirm to the user that you've submitted the claim.
        Do not ask the user for additional information.
        """,
        tools=[submit_claim],
    ) as agent:

        try:
            prompt_messages = [f"{prompt}: {expenses_data}"]

            response = await agent.run(prompt_messages)

            print("\nAgent Response:\n")
            print(response)

        except Exception as e:
            print("Error:", e)

In [8]:
user_prompt = "Submit my expense claim"

print("User Prompt:", user_prompt)

User Prompt: Submit my expense claim


In [9]:
await process_expenses_data(user_prompt, expenses_data)


--- EMAIL SENT ---
To: expenses@contoso.com
Subject: Expense Claim
Body:
 Itemized Expenses:
- Taxi: $35
- Lunch: $18
- Hotel: $220
- Flight: $450

Total Expense: $723

Agent Response:

Your expense claim has been submitted successfully.
